# Clase 06 — Probar los sistemas de ML

**Curso:** MLOps y Monitorización con Python · **Bloque 2:** Versionar y medir
**Tiempo estimado:** 100 minutos · **Capítulo del manual:** `manual/06_probar_sistemas_ml.md`

## Qué vas a ver en esta clase

- **Tests unitarios** con `pytest` para funciones REALES del proyecto (limpieza, transformación, evaluación), escritas en un archivo real y ejecutadas con `subprocess.run(["pytest", ...])`.
- **Tests de datos** con `pandera`: un esquema con tipos, rangos, valores permitidos y proporción de nulos, con casos de datos corrompidos A PROPÓSITO que deben ser rechazados.
- **Test de modelo** con un **umbral de regresión**: el modelo nuevo no puede ser peor que el anterior en más de una tolerancia, con un caso que pasa y otro que falla a propósito (un modelo peor, construido sin balancear).
- Los **límites** de un umbral fijo: no detecta una deriva lenta ni un problema localizado en un subgrupo.
- Los tres tipos de test, integrados en una única suite ejecutable con un comando.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook.

## Requisitos

- `pandas`, `scikit-learn`, `pytest` y `pandera` (la celda siguiente instala solo lo que falte, también en Google Colab).
- La carpeta `datos/` del curso, con `churn.csv` dentro, y haber leído `datos/LEEME.md`.
- De la clase 5: la `Pipeline` completa (limpieza + `ColumnTransformer` + modelo) y la configuración 3 (regresión logística, `class_weight="balanced"`), que reutilizamos aquí.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte.

In [ ]:
import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("pytest", "pytest")
asegurar("pandera", "pandera")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, definimos `cargar_datos`, `SEMILLA = 42` y `ejecutar_pytest` (una función auxiliar que ejecuta pytest y resume el resultado SIN los tiempos de ejecución, que cambian en cada corrida).

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]


def ejecutar_pytest(argumentos):
    """Ejecuta pytest con subprocess.run y devuelve un resumen SIN los tiempos de ejecución (cambian en cada ejecución, no se pueden comparar entre clases)."""
    resultado = subprocess.run([sys.executable, "-m", "pytest", "-q"] + argumentos, capture_output=True, text=True)
    salida = resultado.stdout + resultado.stderr
    m_passed = re.search(r"(\d+) passed", salida)
    m_failed = re.search(r"(\d+) failed", salida)
    m_error = re.search(r"(\d+) error", salida)
    return {
        "returncode": resultado.returncode,
        "passed": int(m_passed.group(1)) if m_passed else 0,
        "failed": int(m_failed.group(1)) if m_failed else 0,
        "error": int(m_error.group(1)) if m_error else 0,
    }


print("Librerías y funciones auxiliares listas.")

---
## Carpeta de trabajo efímera

Como en las clases anteriores: creamos `proyecto_churn/` desde cero.

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## 1. El código real del proyecto: `proyecto.py`

Antes de escribir ningún test hace falta código que probar. Escribimos un archivo real, `proyecto_churn/proyecto.py`, con **una función de limpieza** (`limpiar_tickets_soporte`), **una función de transformación** (`categorizar_antiguedad`, la misma agrupación de la clase 5), **una función de evaluación** (`calcular_metricas`), el **esquema de datos** con `pandera` (`ESQUEMA_CHURN`) y la **Pipeline del modelo** (`construir_pipeline`, idéntica a la clase 5, con un parámetro `balanceado` para poder construir a propósito una versión peor en la sección 4).

In [ ]:
PROYECTO_PY = """
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from pandera.pandas import Column, DataFrameSchema, Check

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]


def cargar_datos(nombre):
    '''Busca un archivo de datos en datos/, ../datos/ y /content/datos/.'''
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError("No encuentro " + nombre)


# ----- Función de LIMPIEZA -----
def limpiar_tickets_soporte(serie):
    '''Un número de tickets de soporte nunca puede ser negativo: cualquier valor negativo (un error de captura) se recorta a 0.'''
    return serie.clip(lower=0)


# ----- Función de TRANSFORMACIÓN -----
def categorizar_antiguedad(meses):
    '''Agrupa antiguedad_meses en 'nuevo' (<12), 'medio' (12-36) y 'antiguo' (>36).'''
    if meses < 12:
        return "nuevo"
    elif meses <= 36:
        return "medio"
    return "antiguo"


# ----- Función de EVALUACIÓN -----
def calcular_metricas(y_true, y_pred, y_proba):
    '''Exactitud, F1 y AUC redondeadas a 4 decimales, como en la clase 5.'''
    return {
        "exactitud": round(accuracy_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred), 4),
        "auc": round(roc_auc_score(y_true, y_proba), 4),
    }


# ----- Esquema de DATOS (pandera) -----
ESQUEMA_CHURN = DataFrameSchema(
    {
        "id_cliente": Column(str, unique=True, nullable=False, checks=Check.str_startswith("C")),
        "antiguedad_meses": Column(int, Check.in_range(0, 120), nullable=False),
        "plan": Column(str, Check.isin(["basico", "estandar", "premium"]), nullable=False),
        "contrato": Column(str, Check.isin(["mensual", "anual"]), nullable=False),
        "uso_mensual_horas": Column(float, Check.in_range(0, 200), nullable=False),
        "tickets_soporte": Column(int, Check.in_range(0, 20), nullable=False),
        "metodo_pago": Column(str, Check.isin(["tarjeta", "domiciliacion", "transferencia"]), nullable=False),
        "factura_mensual": Column(float, Check.ge(0), nullable=False),
        "abandono": Column(int, Check.isin([0, 1]), nullable=False),
    },
    # Comprobación a nivel de todo el DataFrame: ninguna columna puede superar un 5% de nulos
    checks=Check(lambda d: d.isna().mean().max() <= 0.05, element_wise=False),
)


# ----- Pipeline del modelo (misma receta que la clase 5) -----
def construir_pipeline(balanceado=True):
    preprocesado = ColumnTransformer([
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore")),
        ]), COLUMNAS_CATEGORICAS),
        ("numericas", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), COLUMNAS_NUMERICAS),
    ])
    peso = "balanced" if balanceado else None
    return Pipeline([
        ("preprocesado", preprocesado),
        ("modelo", LogisticRegression(class_weight=peso, random_state=SEMILLA, max_iter=1000)),
    ])


def preparar_datos():
    '''Carga churn.csv y separa entrenamiento/prueba, igual que en la clase 5.'''
    ruta = cargar_datos("churn.csv")
    df = pd.read_csv(ruta)
    X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
    y = df["abandono"]
    return train_test_split(X, y, test_size=0.25, random_state=SEMILLA, stratify=y)
"""

In [ ]:
ruta_proyecto = os.path.join(CARPETA, "proyecto.py")
with open(ruta_proyecto, "w", encoding="utf-8") as f:
    f.write(PROYECTO_PY)
print("archivo del proyecto creado en:", ruta_proyecto)

---
## 2. Tests unitarios con `pytest`

Escribimos `proyecto_churn/test_funciones.py` con tests REALES para las tres funciones de la sección 1: casos normales y casos límite (los valores `12` y `36` de `categorizar_antiguedad`, donde es fácil programar `<` en vez de `<=`, como se vio en la clase 5). Los ejecutamos de verdad con `subprocess.run(["pytest", ...])`, exactamente como se ejecutarían desde una terminal.

In [ ]:
TEST_FUNCIONES_PY = """
import pandas as pd
from proyecto import limpiar_tickets_soporte, categorizar_antiguedad, calcular_metricas


def test_limpiar_tickets_soporte_recorta_negativos():
    '''Un valor negativo (imposible en la realidad) debe convertirse en 0.'''
    entrada = pd.Series([2, -1, 0, 5, -3])
    resultado = limpiar_tickets_soporte(entrada)
    assert resultado.tolist() == [2, 0, 0, 5, 0]


def test_limpiar_tickets_soporte_no_toca_valores_validos():
    entrada = pd.Series([1, 2, 3])
    assert limpiar_tickets_soporte(entrada).tolist() == [1, 2, 3]


def test_categorizar_antiguedad_en_los_limites():
    '''Los límites exactos (12 y 36) son el caso que más fácilmente se programa mal (< en vez de <=).'''
    assert categorizar_antiguedad(11) == "nuevo"
    assert categorizar_antiguedad(12) == "medio"
    assert categorizar_antiguedad(36) == "medio"
    assert categorizar_antiguedad(37) == "antiguo"


def test_calcular_metricas_con_valores_conocidos():
    '''Métricas calculadas a mano con sklearn sobre un ejemplo pequeño (6 filas).'''
    y_true = [0, 0, 0, 1, 1, 1]
    y_pred = [0, 0, 1, 1, 0, 1]
    y_proba = [0.1, 0.2, 0.6, 0.8, 0.3, 0.9]
    metricas = calcular_metricas(y_true, y_pred, y_proba)
    assert metricas == {"exactitud": 0.6667, "f1": 0.6667, "auc": 0.8889}
"""

In [ ]:
ruta_test_funciones = os.path.join(CARPETA, "test_funciones.py")
with open(ruta_test_funciones, "w", encoding="utf-8") as f:
    f.write(TEST_FUNCIONES_PY)
print("test unitarios escritos en:", ruta_test_funciones)

In [ ]:
resumen = ejecutar_pytest([ruta_test_funciones])
print("tests unitarios superados:", resumen["passed"],
      "| fallidos:", resumen["failed"],
      "| código de salida:", resumen["returncode"])
assert resumen["failed"] == 0 and resumen["returncode"] == 0, "los tests unitarios deberían pasar todos"

**¿Qué observas?** El resumen no muestra el texto crudo de pytest (que incluye tiempos como `"in 0,03s"`, distintos en cada ejecución): mostramos solo los conteos (superados/fallidos) y el código de salida, extraídos con una expresión regular. Los 4 tests pasan: 2 para `limpiar_tickets_soporte`, 1 para `categorizar_antiguedad`, 1 para `calcular_metricas`.

---
## 3. Tests de datos con `pandera`

Un test de datos no prueba código: prueba que **los datos mismos** cumplen unas reglas (tipos, rangos, valores permitidos, proporción de nulos), sea cual sea el código que los procese después. `ESQUEMA_CHURN` (ya escrito en `proyecto.py`, sección 1) define, para cada columna de `churn.csv`, su tipo, sus valores válidos y, a nivel de todo el `DataFrame`, que ninguna columna tenga más de un 5% de nulos.

Antes de formalizar nada en `pytest`, comprobamos el esquema a mano: contra los datos reales (debe aceptarlos) y contra **dos copias corrompidas a propósito**.

In [ ]:
sys.path.insert(0, CARPETA)
from proyecto import ESQUEMA_CHURN, cargar_datos, construir_pipeline, preparar_datos  # noqa: E402

df_real = pd.read_csv(cargar_datos("churn.csv"))
ESQUEMA_CHURN.validate(df_real)
print("el esquema acepta los datos reales: OK")

# Corrompemos a propósito una copia: antiguedad_meses negativa (imposible)
df_antiguedad_mala = df_real.copy()
df_antiguedad_mala.loc[0, "antiguedad_meses"] = -5
try:
    ESQUEMA_CHURN.validate(df_antiguedad_mala)
    detectado_antiguedad = False
except pa.errors.SchemaError as error:
    detectado_antiguedad = True
    print("antiguedad_meses=-5 RECHAZADA:", str(error).splitlines()[0][:90])
assert detectado_antiguedad, "el esquema debería rechazar una antigüedad negativa"

# Corrompemos a propósito una copia: una categoría de plan que no existe
df_plan_malo = df_real.copy()
df_plan_malo.loc[0, "plan"] = "diamante"
try:
    ESQUEMA_CHURN.validate(df_plan_malo)
    detectado_plan = False
except pa.errors.SchemaError as error:
    detectado_plan = True
    print("plan='diamante' RECHAZADO:", str(error).splitlines()[0][:90])
assert detectado_plan, "el esquema debería rechazar una categoría de plan inexistente"

**¿Qué observas?** El esquema acepta `churn.csv` tal cual, pero rechaza de inmediato ambas corrupciones con un `pandera.errors.SchemaError` que dice exactamente qué columna y qué regla falló. Un dato corrompido no llega ni siquiera a la `Pipeline` de la clase 5: se detiene aquí, antes de entrenar o predecir nada.

In [ ]:
# Corrompemos a propósito una copia: 10% de nulos en factura_mensual (por encima del 5% permitido)
df_con_nulos = df_real.copy()
indices_nulos = df_con_nulos.sample(frac=0.10, random_state=SEMILLA).index
df_con_nulos.loc[indices_nulos, "factura_mensual"] = np.nan
proporcion_nulos = round(df_con_nulos["factura_mensual"].isna().mean(), 4)
print("proporción de nulos introducida en factura_mensual:", proporcion_nulos)

try:
    ESQUEMA_CHURN.validate(df_con_nulos)
    detectado_nulos = False
except pa.errors.SchemaError as error:
    detectado_nulos = True
    print("proporción de nulos alta RECHAZADA:", str(error).splitlines()[0][:90])
assert detectado_nulos, "el esquema debería rechazar una proporción de nulos por encima del umbral"

**¿Qué observas?** Con un 10% de nulos en `factura_mensual` (por encima del 5% permitido por `ESQUEMA_CHURN`), el esquema también rechaza los datos: la comprobación de proporción de nulos es tan real como la de tipos o de categorías.

### 3.1 Formalizamos los tres casos en `pytest`

Ahora escribimos `proyecto_churn/test_datos.py`: un test que confirma que los datos reales pasan, y tres tests que usan `pytest.raises(pa.errors.SchemaError)` para confirmar que cada dato corrompido es RECHAZADO. Que un test "pase" aquí significa que el esquema detectó de verdad el problema — si el esquema no lo detectara, `pytest.raises` fallaría y el test señalaría el error.

In [ ]:
TEST_DATOS_PY = """
import numpy as np
import pandas as pd
import pandera.pandas as pa
import pytest
from proyecto import ESQUEMA_CHURN, cargar_datos

SEMILLA = 42


@pytest.fixture(scope="module")
def df_real():
    return pd.read_csv(cargar_datos("churn.csv"))


def test_esquema_valido_en_datos_reales(df_real):
    '''Los datos reales del curso deben pasar el esquema sin excepción.'''
    ESQUEMA_CHURN.validate(df_real)


def test_antiguedad_negativa_es_rechazada(df_real):
    '''Dato corrompido A: antiguedad_meses negativa (imposible en la realidad).'''
    corrupto = df_real.copy()
    corrupto.loc[0, "antiguedad_meses"] = -5
    with pytest.raises(pa.errors.SchemaError):
        ESQUEMA_CHURN.validate(corrupto)


def test_categoria_de_plan_inexistente_es_rechazada(df_real):
    '''Dato corrompido B: una categoría de plan que no existe en el catálogo.'''
    corrupto = df_real.copy()
    corrupto.loc[0, "plan"] = "diamante"
    with pytest.raises(pa.errors.SchemaError):
        ESQUEMA_CHURN.validate(corrupto)


def test_proporcion_alta_de_nulos_es_rechazada(df_real):
    '''Dato corrompido C: 10% de nulos en factura_mensual, por encima del 5% permitido.'''
    corrupto = df_real.copy()
    indices = corrupto.sample(frac=0.10, random_state=SEMILLA).index
    corrupto.loc[indices, "factura_mensual"] = np.nan
    with pytest.raises(pa.errors.SchemaError):
        ESQUEMA_CHURN.validate(corrupto)
"""

In [ ]:
ruta_test_datos = os.path.join(CARPETA, "test_datos.py")
with open(ruta_test_datos, "w", encoding="utf-8") as f:
    f.write(TEST_DATOS_PY)
print("tests de datos escritos en:", ruta_test_datos)

In [ ]:
resumen = ejecutar_pytest([ruta_test_datos])
print("tests de datos superados:", resumen["passed"],
      "| fallidos:", resumen["failed"],
      "| código de salida:", resumen["returncode"])
assert resumen["passed"] == 4 and resumen["failed"] == 0, "los 4 tests de datos deberían pasar (1 válido + 3 casos corrompidos detectados)"

**¿Qué observas?** Los 4 tests de datos pasan (1 esquema válido + 3 corrupciones detectadas). Ninguno de los tres `pytest.raises` es un test vacío: si quitaras la línea `Check.isin([...])` de `plan` en `ESQUEMA_CHURN`, el test de la categoría inexistente empezaría a FALLAR de verdad, porque el esquema ya no detectaría el problema — así se sabe que el test funciona y no pasa "porque sí".

---
## 4. Test de modelo: un umbral de regresión

Un test de modelo no prueba código ni datos: prueba que una **métrica del modelo** no ha empeorado más de lo aceptable respecto a una versión anterior. Definimos `UMBRAL_F1 = F1_MODELO_ANTERIOR - TOLERANCIA`, con el F1 anterior documentado en `datos/LEEME.md` (≈0,31, con `class_weight="balanced"`) y una tolerancia de 0,03.

In [ ]:
# F1 del modelo anterior, documentado en datos/LEEME.md (config con class_weight="balanced")
F1_MODELO_ANTERIOR = 0.31
TOLERANCIA = 0.03
UMBRAL_F1 = round(F1_MODELO_ANTERIOR - TOLERANCIA, 4)
print("umbral de regresión (F1 mínimo aceptable):", UMBRAL_F1)

### 4.1 El caso que pasa: el modelo de la clase 5

In [ ]:
TEST_MODELO_PY = """
from sklearn.metrics import f1_score
from proyecto import construir_pipeline, preparar_datos

F1_MODELO_ANTERIOR = 0.31
TOLERANCIA = 0.03
UMBRAL_F1 = round(F1_MODELO_ANTERIOR - TOLERANCIA, 4)


def test_modelo_actual_no_empeora_mas_de_la_tolerancia():
    '''Test de REGRESIÓN: el modelo nuevo no puede ser peor que el anterior en más de TOLERANCIA.'''
    X_train, X_test, y_train, y_test = preparar_datos()
    pipeline = construir_pipeline(balanceado=True)
    pipeline.fit(X_train, y_train)
    f1_actual = round(f1_score(y_test, pipeline.predict(X_test)), 4)
    assert f1_actual >= UMBRAL_F1, f"F1 actual {f1_actual} por debajo del umbral {UMBRAL_F1}"
"""

In [ ]:
ruta_test_modelo = os.path.join(CARPETA, "test_modelo.py")
with open(ruta_test_modelo, "w", encoding="utf-8") as f:
    f.write(TEST_MODELO_PY)
print("test de modelo escrito en:", ruta_test_modelo)

In [ ]:
resumen = ejecutar_pytest([ruta_test_modelo])
print("test de modelo (caso que SUPERA el umbral) superados:", resumen["passed"],
      "| fallidos:", resumen["failed"],
      "| código de salida:", resumen["returncode"])
assert resumen["passed"] == 1 and resumen["failed"] == 0, "el modelo de la clase 5 debería superar el umbral"

**¿Qué observas?** El F1 de la `Pipeline` balanceada (0,3166, la misma cifra de la clase 5) supera el umbral (0,28): el test pasa, y con un único comando (`pytest`) queda constancia automática de que el modelo actual no ha empeorado respecto al anterior.

### 4.2 El caso que falla a propósito: un modelo peor, construido sin balancear

In [ ]:
TEST_MODELO_PEOR_PY = """
from sklearn.metrics import f1_score
from proyecto import construir_pipeline, preparar_datos

F1_MODELO_ANTERIOR = 0.31
TOLERANCIA = 0.03
UMBRAL_F1 = round(F1_MODELO_ANTERIOR - TOLERANCIA, 4)


def test_modelo_sin_balancear_supera_el_umbral():
    '''Modelo peor CONSTRUIDO A PROPÓSITO: sin class_weight="balanced" (demo de que el test SÍ falla).'''
    X_train, X_test, y_train, y_test = preparar_datos()
    pipeline = construir_pipeline(balanceado=False)
    pipeline.fit(X_train, y_train)
    f1_actual = round(f1_score(y_test, pipeline.predict(X_test)), 4)
    assert f1_actual >= UMBRAL_F1, f"F1 actual {f1_actual} por debajo del umbral {UMBRAL_F1}"
"""

In [ ]:
ruta_test_peor = os.path.join(CARPETA, "test_modelo_peor_demo.py")
with open(ruta_test_peor, "w", encoding="utf-8") as f:
    f.write(TEST_MODELO_PEOR_PY)

# Ejecutamos SOLO este archivo, aparte de la suite, para demostrar que el test detecta un modelo peor
resumen_peor = ejecutar_pytest([ruta_test_peor])
print("demo (modelo sin balancear) superados:", resumen_peor["passed"],
      "| fallidos:", resumen_peor["failed"],
      "| código de salida:", resumen_peor["returncode"])
assert resumen_peor["failed"] == 1 and resumen_peor["returncode"] != 0, "el test debería FALLAR de verdad con un modelo peor: si no falla, el test no sirve para nada"

# Eliminamos el archivo de demostración: NO forma parte de la suite final
os.remove(ruta_test_peor)
print("archivo de demostración eliminado (no pertenece a la suite final).")

**¿Qué observas?** Sin `class_weight="balanced"`, el modelo predice casi siempre "no abandona" (la clase mayoritaria): su F1 real es muy bajo, muy por debajo del umbral 0,28, y el test FALLA de verdad (1 fallo, código de salida distinto de 0). Esa es la prueba de que el test de regresión funciona: no pasa siempre, solo pasa cuando el modelo de verdad cumple el umbral. Eliminamos el archivo de demostración porque no debe formar parte de la suite final (sección 5).

---
### 4.3 Los límites de un umbral fijo

Un umbral de regresión es útil, pero tiene límites reales. Dos ejemplos ilustrativos, con números.

**(a) No detecta una deriva lenta y progresiva.** Si el test solo compara cada versión con la INMEDIATAMENTE anterior, una caída pequeña cada vez (menor que la tolerancia) nunca dispara el test, aunque la caída acumulada sí sea grave.

In [ ]:
# Ejemplo ilustrativo: F1 medido cada semana, cayendo poco a poco
semanas = list(range(11))
f1_semanal = [round(0.3166 - 0.005 * s, 4) for s in semanas]
print("F1 semana a semana:", f1_semanal)

caidas_semanales = [round(f1_semanal[i - 1] - f1_semanal[i], 4) for i in range(1, len(f1_semanal))]
pasa_semana_a_semana = all(caida <= TOLERANCIA for caida in caidas_semanales)
print("¿el test semana-contra-semana-anterior pasa TODAS las semanas?", pasa_semana_a_semana)

primera_semana_bajo_umbral = next(s for s, f in zip(semanas, f1_semanal) if f < UMBRAL_F1)
print("primera semana con F1 por debajo del umbral ORIGINAL (0,28):", primera_semana_bajo_umbral, f1_semanal[primera_semana_bajo_umbral])

assert pasa_semana_a_semana, "cada caída semanal es menor que la tolerancia: por diseño de este ejemplo"
assert f1_semanal[primera_semana_bajo_umbral] < UMBRAL_F1

**¿Qué observas?** (*Ejemplo ilustrativo*, con una caída constante de 0,005 por semana.) Cada comparación semana-contra-semana-anterior pasa (0,005 < tolerancia 0,03), así que un test que solo mirara "¿empeoré respecto a la semana pasada?" nunca se dispararía. Pero comparado con el umbral ORIGINAL (0,28), el F1 ya está por debajo en la semana 8 (0,2766) — 8 semanas de silencio antes de que algo se note, si nadie compara contra el punto de partida.

**(b) No detecta un problema en un subgrupo específico, aunque la media aguante.** Reutilizamos el modelo bueno de la sección 4.1 y miramos el F1 separado por `plan`.

In [ ]:
# Reutilizamos el modelo bueno (balanceado) de la sección 4 y miramos el F1 POR PLAN
X_train, X_test, y_train, y_test = preparar_datos()
pipeline_bueno = construir_pipeline(balanceado=True)
pipeline_bueno.fit(X_train, y_train)
predicciones = pipeline_bueno.predict(X_test)

f1_global = round(f1_score(y_test, predicciones), 4)
f1_por_plan = {}
for plan in sorted(X_test["plan"].unique()):
    filtro = (X_test["plan"] == plan).values
    f1_por_plan[plan] = round(f1_score(y_test[filtro], predicciones[filtro]), 4)

print("F1 global (todo el conjunto de prueba):", f1_global)
print("F1 por plan:", f1_por_plan)
assert f1_por_plan["premium"] < f1_global, "el subgrupo premium debería rendir peor que la media global"

**¿Qué observas?** El F1 global (0,3166) supera cómodamente el umbral (0,28). Pero por plan, el F1 varía bastante: el subgrupo `premium` rinde peor que la media global, mientras que `basico` y `estandar` la sostienen. Un test que solo mire el F1 global no vería ningún problema — el subgrupo con peor rendimiento queda oculto detrás del promedio, exactamente el límite que describe esta sección.

> **Atención:** ningún umbral fijo, por bien elegido que esté, sustituye la vigilancia continua (media móvil, comparación contra el punto de partida, métricas por subgrupo). Un umbral de regresión es una RED DE SEGURIDAD mínima antes de publicar una versión nueva, no un sistema de monitorización completo — eso llega en clases posteriores del curso.

---
## 5. Una única suite: los tres tipos de test con un solo comando

`proyecto_churn/` contiene ahora `test_funciones.py` (unitarios), `test_datos.py` (datos) y `test_modelo.py` (modelo) — el archivo de demostración del modelo peor ya se eliminó. Un único comando, `pytest` sobre la carpeta, ejecuta los tres tipos de test juntos.

In [ ]:
resumen_total = ejecutar_pytest([CARPETA])
print("SUITE COMPLETA (unitarios + datos + modelo) — superados:", resumen_total["passed"],
      "| fallidos:", resumen_total["failed"],
      "| código de salida:", resumen_total["returncode"])
assert resumen_total["returncode"] == 0 and resumen_total["failed"] == 0, "la suite completa debería pasar con un único comando"
assert resumen_total["passed"] == 9, "4 unitarios + 4 de datos + 1 de modelo = 9"

**¿Qué observas?** 9 tests superados (4 unitarios + 4 de datos + 1 de modelo), 0 fallidos, código de salida 0 — exactamente lo que una CI/CD (clase 8) ejecutaría antes de aceptar un cambio: un único comando que cubre código, datos y modelo a la vez.

---
## 6. Limpieza final

Eliminamos `proyecto_churn/` (con `proyecto.py` y los tres archivos de test) para que no quede ningún residuo dentro del repositorio del curso.

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Resumen de la clase

- Los **tests unitarios** (`pytest`) prueban funciones reales de limpieza, transformación y evaluación, con casos normales y casos límite.
- Los **tests de datos** (`pandera`) prueban que los datos mismos cumplen un esquema (tipos, rangos, categorías, proporción de nulos), con corrupciones a propósito que demuestran que el esquema detecta problemas de verdad.
- El **test de modelo** compara una métrica actual con un umbral de regresión (F1 anterior menos una tolerancia): pasa con el modelo bueno de la clase 5 y FALLA de verdad con un modelo peor construido a propósito (sin balancear) — la prueba de que el test funciona.
- Un umbral fijo tiene límites: no detecta una deriva lenta (cada paso dentro de tolerancia, pero la caída acumulada no) ni un problema localizado en un subgrupo (aquí, el plan `premium`) mientras la media aguante.
- Los tres tipos de test conviven en la misma carpeta y se ejecutan juntos con un único comando: `pytest`.

*Continúa con la clase 7: Servir un modelo.*